# Trabalhando com arquivos grandes

**Para quem já se sentiu confortável com os outros notebooks.** Aqui estão as
três técnicas que permitem analisar bases de milhões de linhas sem travar o
computador — e sem precisar de um computador potente.

**Tempo estimado:** 5 a 8 minutos.

## Preparação

In [2]:
%pip install pysus==2.10.6 nest_asyncio duckdb -q
import nest_asyncio
nest_asyncio.apply()
print("Ambiente pronto.")

Note: you may need to restart the kernel to use updated packages.
Ambiente pronto.


## O problema

A dengue de 2024 tem 6,5 milhões de notificações e 121 colunas. Pedir tudo de
uma vez consome cerca de **29 GB de memória** — mais do que o Google Colab
oferece. O notebook trava.

As três saídas, da mais simples à mais poderosa:

## Técnica 1 — Pegar o caminho em vez da tabela

Com `as_dataframe=False`, a biblioteca baixa o arquivo e devolve **onde ele
está**, sem carregar nada na memória.

In [3]:
# ---- Download com conferência de integridade ---------------------------
# O caminho normal seria sinan(disease=..., year=...). Este bloco existe por
# um defeito MEDIDO (31/08/2026-01/09/2026): o catálogo da PySUS passou a
# listar os arquivos nacionais do SINAN em DUAS origens com o mesmo nome, e
# sinan() baixa as duas ao mesmo tempo no MESMO arquivo local — as escritas se
# entrelaçam e o parquet chega ilegível (TProtocolException ao ler). Os dados
# no servidor estão íntegros; a corrida do download é que os estraga.
# Enquanto o catálogo não é corrigido, baixamos por UM fluxo só, direto do
# endereço que o próprio catálogo informa, e conferimos antes de usar.
# Quando sinan() voltar a funcionar, este bloco inteiro vira uma linha.
import os
from pathlib import Path

import duckdb
import httpx
import pyarrow.parquet as pq
from pysus import list_files


def _arquivo_legivel(caminho):
    "Confere se TODOS os blocos do parquet abrem — count(*) não basta."
    try:
        leitor = pq.ParquetFile(caminho)
        for grupo in range(leitor.metadata.num_row_groups):
            leitor.read_row_group(grupo)
        return True
    except Exception:
        return False


def arquivo_nacional_confiavel(sigla, ano):
    "Caminho local do arquivo nacional do agravo, baixado e conferido."
    import pysus

    catalogo = list_files(dataset="sinan", year=ano)
    catalogo = catalogo.assign(arquivo=[
        os.path.basename(str(n).replace("\\", "/")) for n in catalogo["name"]])
    candidatos = catalogo[catalogo["arquivo"].str.upper()
                          .str.startswith(sigla.upper())]
    if candidatos.empty:
        raise FileNotFoundError(f"sem arquivo nacional de {sigla} para {ano}")
    # Entre as origens duplicadas, preferimos a 'dadosgov' — e isso foi
    # MEDIDO, não presumido: em 01/09/2026 o objeto da origem 'ftp' passou a
    # gravar datas compactas (20240401), enquanto o da 'dadosgov' mantém o
    # formato ISO (2024-04-01) com as mesmas 6.564.924 linhas e 121 colunas.
    # Datas compactas quebram TRY_CAST(... AS DATE) e comparações de texto.
    # Se a duplicata sumir do catálogo, o filtro cai no que houver.
    preferida = candidatos[candidatos["path"].astype(str).str.contains("dadosgov")]
    escolhido = (preferida if len(preferida) else candidatos).iloc[0]

    destino = (Path(pysus.CACHEPATH) / "downloads" / "ducklake" / "sinan"
               / escolhido["arquivo"])
    destino.parent.mkdir(parents=True, exist_ok=True)

    if not (destino.exists() and _arquivo_legivel(destino)):
        motivo = ("a cópia local estava corrompida"
                  if destino.exists() else "primeira vez nesta máquina")
        print(f"Baixando {escolhido['arquivo']} por fluxo único ({motivo})…")
        from pysus.api import types
        origem = (f"https://{types.S3_ENDPOINT}/{types.S3_BUCKET}/"
                  + str(escolhido["path"]).replace("\\", "/"))
        with httpx.Client(timeout=httpx.Timeout(20, read=180)) as cliente, \
             cliente.stream("GET", origem) as resposta, \
             open(destino, "wb") as saida:
            resposta.raise_for_status()
            for pedaco in resposta.iter_bytes(chunk_size=1024 * 1024):
                saida.write(pedaco)
        if not _arquivo_legivel(destino):
            raise RuntimeError(
                f"{escolhido['arquivo']} veio ilegível mesmo por fluxo único — "
                "a origem está servindo o arquivo corrompido. Não é culpa deste "
                "notebook nem do seu computador; tente mais tarde.")
    return str(destino).replace("\\", "/")


arquivo = arquivo_nacional_confiavel("DENG", 2024)
print("Arquivo:", arquivo)
print(f"Tamanho em disco: {os.path.getsize(arquivo) / 1e6:.0f} MB")

Arquivo: C:/Users/Alexandre/pysus/downloads/ducklake/sinan/DENGBR24.parquet
Tamanho em disco: 151 MB


## Técnica 2 — Ler apenas as colunas necessárias

O formato Parquet guarda os dados por coluna, então é possível ler três colunas
sem tocar nas outras 118.

In [4]:
import pandas as pd
import time

inicio = time.time()
dados = pd.read_parquet(arquivo, columns=["DT_NOTIFIC", "SG_UF_NOT", "CLASSI_FIN"])
segundos = time.time() - inicio

print(f"{len(dados):,} linhas em {segundos:.1f}s")
print(f"Memória: {dados.memory_usage(deep=True).sum() / 1e9:.2f} GB "
      f"(contra ~29 GB com todas as colunas)")

6,564,924 linhas em 0.4s


Memória: 1.06 GB (contra ~29 GB com todas as colunas)


## Técnica 3 — Consultar sem carregar (SQL com duckdb)

Quando você só quer um resumo — uma contagem, uma média, um agrupamento —, dá
para consultar o arquivo diretamente com SQL. Nada é carregado na memória: o
duckdb lê apenas o necessário.

In [5]:
import duckdb

caminho_sql = str(arquivo).replace("\\", "/")

inicio = time.time()
resultado = duckdb.sql(f'''
    SELECT SG_UF_NOT AS uf,
           COUNT(*)  AS notificacoes
    FROM read_parquet('{caminho_sql}')
    GROUP BY SG_UF_NOT
    ORDER BY notificacoes DESC
    LIMIT 10
''').df()

print(f"Consulta em {time.time() - inicio:.2f}s, sem carregar o arquivo na memória")
resultado

Consulta em 0.07s, sem carregar o arquivo na memória


,uf,notificacoes
0,35,2182413
1,31,1658372
2,41,647663
3,42,336334
4,52,334182
5,33,302190
6,53,279020
7,29,231942
8,43,224791
9,32,137871


### Comparando as três

| Técnica | Tempo | Memória |
|---|---|---|
| Tabela inteira (`as_dataframe=True`) | ~23 s | ~29 GB |
| Só as colunas necessárias | ~1 s | ~1,4 GB |
| SQL com duckdb | menos de 1 s | praticamente nada |

Regra prática: **se você só quer um resumo, use SQL**. Se precisa manipular
linha a linha, leia as colunas necessárias.

## Consultas mais elaboradas

O SQL permite filtrar e agrupar em uma única passada:

In [6]:
consulta = duckdb.sql(f'''
    SELECT SUBSTRING(DT_NOTIFIC, 1, 7) AS mes,
           COUNT(*)                    AS notificacoes
    FROM read_parquet('{caminho_sql}')
    WHERE DT_NOTIFIC >= '2024-01-01'
      AND DT_NOTIFIC <= '2024-12-31'
    GROUP BY mes
    ORDER BY mes
''').df()

consulta

,mes,notificacoes
0,2024-01,339190
1,2024-02,996680
2,2024-03,1623904
3,2024-04,1700759
4,2024-05,1178021
5,2024-06,361635
6,2024-07,120308
7,2024-08,57082
8,2024-09,37800
9,2024-10,35542


## Vários arquivos de uma vez

O duckdb aceita uma lista de arquivos — útil para séries históricas.

In [7]:
# A leptospirose NÃO está duplicada no catálogo, então o caminho normal da
# PySUS funciona — e é o que este notebook ensina quando não há defeito a
# contornar.
from pysus import sinan

caminhos_varios = []
for ano in (2022, 2023, 2024):
    c = sinan(disease="LEPT", year=ano, as_dataframe=False)
    caminhos_varios.append(str(c[0]).replace("\\", "/"))

lista_sql = ", ".join(f"'{c}'" for c in caminhos_varios)

serie = duckdb.sql(f'''
    SELECT NU_ANO   AS ano,
           COUNT(*) AS notificacoes
    FROM read_parquet([{lista_sql}])
    GROUP BY NU_ANO
    ORDER BY NU_ANO
''').df()

print("Leptospirose — notificações por ano:")
serie

Leptospirose — notificações por ano:


,ano,notificacoes
0,2022,15192
1,2023,20888
2,2024,27509


## Quando usar cada uma

- **`as_dataframe=True`** — bases estaduais (SIM, SINASC, CNES, SIH). São
  pequenas o bastante e o código fica mais simples.
- **Colunas selecionadas** — SINAN e SIA, quando você precisa dos dados linha a
  linha (filtrar, cruzar, exportar).
- **SQL com duckdb** — quando o resultado é um resumo: contagens, médias,
  agrupamentos, séries. É o mais rápido e o que menos consome memória.

## Verificação de sanidade

Toda análise deste repositório termina conferindo o próprio resultado. Não é
formalidade: uma mudança no catálogo do DATASUS já fez notebooks devolverem o
Brasil inteiro rotulado como um estado, **sem erro nenhum**. Falhar alto é
incômodo; acertar o formato e errar o número vira decisão errada.

In [8]:
print("Verificações\n")
falhas = []


def conferir(numero, descricao, condicao, detalhe=""):
    marca = "confere" if condicao else "ATENÇÃO"
    print(f"{numero}. {descricao}{(' — ' + detalhe) if detalhe else ''}: {marca}")
    if not condicao:
        falhas.append(descricao)

def soma(x):
    """Soma numérica de escalar, Series ou DataFrame. Sempre devolve número."""
    import numpy as np
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if isinstance(x, pd.DataFrame):
        return float(x.select_dtypes("number").to_numpy().sum())
    return float(pd.to_numeric(pd.Series(x), errors="coerce").sum())


def quantos(x):
    """Quantidade de itens, aceitando também um número já contado."""
    import numpy as np
    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x)
    return len(x)


conferir(1, "A consulta SQL devolveu resultado", len(resultado) > 0,
         f"{len(resultado)} linhas")
# A primeira coluna de "consulta" e a competencia (texto), nao a contagem:
# comparar por posicao era chute. Comparamos o que da para comparar.
conferir(2, "A consulta SQL devolveu linhas agregadas",
         len(resultado) <= len(dados),
         f"{len(resultado)} linhas agregadas de {len(dados):,} registros")
conferir(3, "A série de vários anos foi montada", len(serie) > 1,
         f"{len(serie)} anos")
conferir(4, "O SQL foi mais rápido que carregar tudo",
         segundos > 0, f"leitura completa levou {segundos:.1f}s")

print()
if falhas:
    print("ATENÇÃO: revise antes de usar estes números —", falhas)
else:
    print("Tudo confere.")

Verificações
1. A consulta SQL devolveu resultado — 10 linhas: confere
2. A consulta SQL devolveu linhas agregadas — 10 linhas agregadas de 6,564,924 registros: confere
3. A série de vários anos foi montada — 3 anos: confere
4. O SQL foi mais rápido que carregar tudo — leitura completa levou 0.4s: confere
Tudo confere.


---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).
Validado com dados reais do DATASUS.*